# 01 - Exploring the data

**Phase 1 of `report/project-plan.md`.** This notebook does not train anything. Its job is
to produce the numbers and figures that justify the decisions made later, so that the report
can say *why* we picked something instead of just stating that we did.

What it feeds into the report:

| Section | What this notebook provides |
|---|---|
| **(a)** feature format + preprocessing | vocabulary sizes before/after `min_df` and `max_features`, why frequency alone is not a usable signal, and a measured justification for `ngram_range=(1, 2)` |
| **(b)** model description | the review-length distribution that justifies BERT's `max_length` |
| **(d)** metrics | the class-balance figure that justifies reporting macro F1 instead of accuracy |
| **(e)** error analysis | the negation and brand-name failure modes to go looking for |
| **(f)** feature-format comparison | section 4.3 - evidence from the data for why a contextual model should beat bag-of-words here |

Figures are written to `results/figures/` as 200 dpi PNGs, ready to drop into the report.

**Before committing:** `Kernel` -> `Restart & Clear Output`. Notebook outputs make git diffs
unreadable (see `notebooks/README.md`).

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# src/ holds load_data.py, which knows where data/ is.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from load_data import load_test, load_train  # noqa: E402

# Same seed as every script in src/ - part (b) asks about reproducibility.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("figures ->", FIG_DIR)

## Figure style

One style block for every figure so they look like a set in the report rather than five
unrelated plots.

The two classes get two fixed colours - **blue = negative, orange = positive** - and keep
them in every figure. Blue/orange is deliberate: red/green is the obvious choice for
sentiment but is the one pair that red-green colourblind readers cannot separate, and the
report gets printed.

In [ ]:
# Fixed colours: one per class, used identically in every figure below.
NEG, POS = "#2a78d6", "#eb6834"
CLASS_COLOR = {0: NEG, 1: POS}
CLASS_NAME = {0: "negative", 1: "positive"}

# Chart chrome: everything that is not data recedes.
INK, INK_MUTED, GRID, AXIS = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.dpi": 110,          # on-screen; savefig uses 200 below
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_MUTED,
    "axes.titlecolor": INK,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "lines.linewidth": 2,
    "font.size": 10,
})


def save(fig, name):
    # Write a figure to results/figures/ and report the path.
    path = FIG_DIR / name
    fig.savefig(path)
    print("wrote", path.relative_to(PROJECT_ROOT))

## 1. Load the data and check it

`data/` is read-only. Nothing here modifies it - this section only reports what is in there,
including the parts that are inconvenient.

In [ ]:
train_df = load_train()
test_df = load_test()

print(f"train : {len(train_df):,} rows, columns = {list(train_df.columns)}")
print(f"test  : {len(test_df):,} rows, columns = {list(test_df.columns)}")
print(f"\nmissing values: {train_df.isna().sum().to_dict()}")
print(f"empty or whitespace-only reviews: {(train_df['reviews'].str.strip() == '').sum()}")
print(f"label values present: {sorted(int(v) for v in train_df['sentiments'].unique())}")

### 1.1 Duplicate reviews

Two things worth checking before trusting any validation number, because both inflate it.

In [ ]:
dup_rows = train_df.duplicated(subset=["reviews"]).sum()
dup_block = train_df[train_df.duplicated(subset=["reviews"], keep=False)]
n_dup_texts = dup_block["reviews"].nunique()

# If the same text carried two different labels the data would be self-contradictory.
conflicting = (dup_block.groupby("reviews")["sentiments"].nunique() > 1).sum()

overlap = set(train_df["reviews"]) & set(test_df["reviews"])

print(f"repeated rows in train.json          : {dup_rows}")
print(f"  ({len(dup_block)} rows share {n_dup_texts} distinct texts)")
print(f"  of those texts, with conflicting labels: {conflicting}")
print(f"reviews appearing in BOTH train and test: {len(overlap)}")

print("\nmost repeated texts:")
for text, n in dup_block["reviews"].value_counts().head(3).items():
    print(f"  x{n}  {text[:80]}...")

**What this means, and what we are doing about it.**

- **216 repeated rows (151 distinct texts, up to 5 copies each).** No text carries two
  different labels, so the data is at least self-consistent. But `train_test_split` is
  random, so copies of the same review land on both sides of the 80/20 split. The model
  then gets scored on reviews it memorised. The reported validation accuracy is therefore
  slightly optimistic - at most a fraction of a percent at this scale, but the report
  should say so rather than pretend the number is clean.
- **87 reviews appear in both `train.json` and `test.json`.** This is a property of the
  data we were given, not something we introduced, and `data/` is read-only. It is worth
  one sentence in the report: roughly 4.7% of the test predictions are recall rather than
  generalisation.

We are **not** de-duplicating. Dropping rows changes the baseline number the whole group
is comparing against, and that is a decision for the group, not a side effect of an
exploration notebook. Flag it in the report and move on.

## 2. Class balance

This is the figure that motivates every metric choice later, so it goes in the report.

In [ ]:
counts = train_df["sentiments"].value_counts().sort_index()
shares = counts / counts.sum()

for label in counts.index:
    print(f"{label} ({CLASS_NAME[label]}): {counts[label]:,} reviews  ({shares[label]:.1%})")

majority = shares.max()
print(f"\nA classifier that always predicts 'positive' scores {majority:.1%} accuracy")
print("while never identifying a single negative review. This is why we report macro F1")
print("and per-class precision/recall, not accuracy alone.")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 2.2))

ypos = [0, 1]
ax.barh(ypos, counts.values, height=0.62,
        color=[CLASS_COLOR[i] for i in counts.index])

# Direct labels instead of an x-axis the reader has to trace back.
for y, label in zip(ypos, counts.index):
    ax.text(counts[label] + 90, y, f"{counts[label]:,}  ({shares[label]:.0%})",
            va="center", ha="left", fontsize=9, color=INK)

ax.set_yticks(ypos, [f"{i} - {CLASS_NAME[i]}" for i in counts.index], fontsize=9, color=INK)
ax.set_xlim(0, counts.max() * 1.22)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_color(AXIS)
ax.invert_yaxis()
ax.set_title("Training labels are 85% positive\n"
             + r"$\it{n}$ = " + f"{len(train_df):,} reviews in train.json", pad=10)

save(fig, "fig01_class_balance.png")
plt.show()

## 3. How long are the reviews?

BERT truncates its input at a fixed `max_length`, and cost grows with it, so the choice is a
trade-off: too short and we cut the end off reviews, too long and we pay for padding. The
distribution tells us where to put it.

Note the unit. This counts **whitespace-separated words**, but BERT splits into *subword*
tokens and adds `[CLS]`/`[SEP]`, so its token count is higher - about 1.3x for English
product-review text. Below we report both the word count and that estimate, so the
`max_length` choice is made against the number that actually matters.

In [ ]:
train_words = train_df["reviews"].str.split().str.len()
test_words = test_df["reviews"].str.split().str.len()

SUBWORD_RATIO = 1.3   # rough words -> BERT WordPiece tokens, English retail text

print("words per review (train):")
print(train_words.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(1).to_string())
print(f"\nmedian test review: {test_words.median():.0f} words "
      f"(train: {train_words.median():.0f}) - the two sets look alike")

print("\ncoverage if we truncate at max_length:")
print(f"{'max_length':>11} | {'by word count':>14} | {'est. subword tokens':>20}")
for n in (64, 128, 192, 256, 384, 512):
    by_word = (train_words <= n).mean()
    by_sub = (train_words * SUBWORD_RATIO <= n).mean()
    print(f"{n:>11} | {by_word:>13.2%} | {by_sub:>19.2%}")

In [ ]:
fig, (ax_hist, ax_cdf) = plt.subplots(1, 2, figsize=(10, 3.4))

# Left: the shape of the distribution. Clipped at 300 so the long tail does not
# squash everything - the clipping is stated in the axis label, not hidden.
clipped = train_words.clip(upper=300)
ax_hist.hist(clipped, bins=60, color=NEG, alpha=0.85)
ax_hist.axvline(train_words.median(), color=INK, linewidth=1.5, linestyle="--")
ax_hist.text(train_words.median() + 8, ax_hist.get_ylim()[1] * 0.88,
             f"median {train_words.median():.0f}", fontsize=9, color=INK)
ax_hist.set_xlabel("words per review (tail clipped at 300)")
ax_hist.set_ylabel("reviews")
ax_hist.yaxis.grid(True)
ax_hist.set_axisbelow(True)
ax_hist.set_title("Most reviews are short")

# Right: the decision chart. Reads as "pick 256 and you keep 98.8% of reviews whole".
order = np.sort(train_words.values)
cdf = np.arange(1, len(order) + 1) / len(order)
ax_cdf.plot(order, cdf, color=NEG)
ax_cdf.set_xscale("log")
ax_cdf.set_xlim(4, 1100)
ax_cdf.set_ylim(0, 1.02)
ax_cdf.set_xlabel("max_length (words, log scale)")
ax_cdf.set_ylabel("share of reviews kept whole")
ax_cdf.yaxis.grid(True)
ax_cdf.set_axisbelow(True)

# Mark the three candidates on the curve, but label them in the empty upper-left
# rather than on top of each other - above 128 the curve is flat and the points
# are only a few pixels apart.
for n in (128, 256, 512):
    kept = (train_words <= n).mean()
    ax_cdf.plot([n], [kept], marker="o", markersize=8, color=POS,
                markeredgecolor="#fcfcfb", markeredgewidth=2, zorder=3)

lines = "\n".join(f"{n:>3} words -> {(train_words <= n).mean():.1%} kept whole"
                  for n in (128, 256, 512))
ax_cdf.text(0.03, 0.97, lines, transform=ax_cdf.transAxes, va="top", ha="left",
            fontsize=9, color=INK, linespacing=1.5,
            family="DejaVu Sans Mono")

ax_cdf.set_xticks([8, 32, 128, 512], ["8", "32", "128", "512"])
ax_cdf.set_title("256 keeps 98.8% of reviews intact")

fig.suptitle("Review length decides BERT's max_length", x=0.065, ha="left",
             fontsize=12, fontweight="bold", color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
save(fig, "fig02_review_length.png")
plt.show()

**Conclusion for the report (question b): `max_length = 256`.**

At 256 tokens, 98.8% of reviews fit whole by word count, and about 97.7% once the subword
expansion is accounted for. Going to 512 buys roughly another 2% of reviews but doubles the
sequence length, and attention cost grows quadratically with it - so that extra 2% costs
about 4x the compute per batch. Dropping to 128 would truncate about 1 review in 16, and
the end of a review is exactly where the verdict usually sits ("...but it fell apart after
a week"), so truncation there is more damaging than it looks.

### 3.1 Does length differ by class?

Worth a look: if negative reviews were systematically longer, length itself would leak the
label and a model could partly cheat on it.

In [ ]:
for label in (0, 1):
    w = train_words[train_df["sentiments"] == label]
    print(f"{CLASS_NAME[label]:>8}: median {w.median():>5.0f}  mean {w.mean():>6.1f} words")

fig, ax = plt.subplots(figsize=(6.4, 3.0))

bins = np.linspace(0, 250, 50)
for label in (0, 1):
    w = train_words[train_df["sentiments"] == label].clip(upper=250)
    # Each class is scaled to its own size: there are ~6x more positive reviews, so
    # raw counts would only re-state the imbalance. Plotting a percentage rather than
    # a density keeps the y-axis directly readable ("3% of negatives are this long").
    ax.hist(w, bins=bins, weights=np.full(len(w), 100 / len(w)),
            histtype="step", linewidth=2,
            color=CLASS_COLOR[label], label=CLASS_NAME[label])

ax.set_xlabel("words per review (tail clipped at 250)")
ax.set_ylabel("% of that class's reviews")
ax.yaxis.grid(True)
ax.set_axisbelow(True)
ax.legend(title="true label", title_fontsize=9)
ax.set_title("Negative reviews run slightly longer\nshapes overlap heavily - length is not a usable signal", pad=10)

save(fig, "fig03_length_by_class.png")
plt.show()

Negative reviews are a little longer (median 50 words against 41) - people explain
complaints more than they explain satisfaction. But the two distributions sit almost on top
of each other, so length carries nowhere near enough signal to classify on, and it is not
leaking the label. No action needed; one sentence in (a).

## 4. Vocabulary: what preprocessing actually does

Question (a) asks for the exact preprocessing steps and the reasoning. Rather than assert
that `min_df=2` and `max_features=20000` are sensible, measure what they remove.

These are the settings from `src/run_baseline.py`, so the numbers describe the real pipeline.

In [ ]:
SHARED = dict(lowercase=True, stop_words="english")

steps = [
    ("unigrams, no cap", dict(ngram_range=(1, 1))),
    ("+ bigrams, no cap", dict(ngram_range=(1, 2))),
    ("+ min_df=2", dict(ngram_range=(1, 2), min_df=2)),
    ("+ max_features=20000 (the baseline)", dict(ngram_range=(1, 2), min_df=2, max_features=20000)),
]

print(f"{'setting':<38} | {'vocabulary':>10}")
for name, kwargs in steps:
    vec = CountVectorizer(**SHARED, **kwargs).fit(train_df["reviews"])
    print(f"{name:<38} | {len(vec.vocabulary_):>10,}")

# Keep the baseline's vectoriser around - sections 4.1 and 5 reuse it.
baseline_vec = CountVectorizer(**SHARED, ngram_range=(1, 2), min_df=2, max_features=20000)
X_counts = baseline_vec.fit_transform(train_df["reviews"])
vocab = np.array(baseline_vec.get_feature_names_out())
print(f"\nfeature matrix: {X_counts.shape[0]:,} x {X_counts.shape[1]:,}, "
      f"{X_counts.nnz / np.prod(X_counts.shape):.2%} of cells non-zero (very sparse)")

**For the report (question a).** Adding bigrams takes the vocabulary from about 13.8k to
135.6k - a 10x blow-up, because most word pairs occur once and never again. `min_df=2`
discards every one of those singletons, cutting 135,616 down to 26,173 - so **four out of
five candidate n-grams were seen exactly once** in 7,401 reviews, and a linear model cannot
learn a reliable weight from a single observation anyway. `max_features=20000` then trims the
survivors to the 20k most frequent, leaving a feature space that is **14.7% of the
candidates**.

The cost: genuinely rare but meaningful words get thrown out too, which is a concrete
weakness to come back to in the error analysis (e).

The usual justification for paying that 10x cost is **negation**: unigram bag-of-words sees
"not" and "good" as two independent features and cannot represent "not good", whereas the
bigram `not good` is a single feature with its own weight. That is the textbook argument.
Section 4.3 checks whether it actually holds on this dataset - the answer turns out to be
more interesting than the textbook version.

### 4.1 The most frequent words tell you almost nothing

The obvious move is to list the most common words in each class. It is worth doing precisely
because of what it shows.

Counts are normalised **per 100 reviews of that class**, because there are about six times
more positive reviews than negative ones - raw counts would only re-state the class
imbalance.

In [ ]:
y = train_df["sentiments"].values
n_pos, n_neg = int((y == 1).sum()), int((y == 0).sum())

pos_counts = np.asarray(X_counts[y == 1].sum(axis=0)).ravel()
neg_counts = np.asarray(X_counts[y == 0].sum(axis=0)).ravel()

# Unigrams only here - bigrams are rarer and would crowd out the comparison.
is_unigram = np.array([" " not in t for t in vocab])

rate_pos = pos_counts / n_pos * 100
rate_neg = neg_counts / n_neg * 100

top_pos = [i for i in np.argsort(-rate_pos) if is_unigram[i]][:12]
top_neg = [i for i in np.argsort(-rate_neg) if is_unigram[i]][:12]

shared = set(vocab[top_pos]) & set(vocab[top_neg])
print(f"top-12 words shared by both classes: {len(shared)} of 12")
print(sorted(shared))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)

for ax, idx, label in zip(axes, (top_neg, top_pos), (0, 1)):
    rate = (rate_neg if label == 0 else rate_pos)[idx]
    words = vocab[idx]
    ypos = np.arange(len(words))
    ax.barh(ypos, rate, height=0.68, color=CLASS_COLOR[label])
    ax.set_yticks(ypos, words, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("occurrences per 100 reviews")
    ax.xaxis.grid(True)
    ax.set_axisbelow(True)
    ax.set_title(f"{CLASS_NAME[label]} reviews")
    # Words that also make the other class's top 12 are set in bold black; the ones
    # unique to this class stay muted, so the overlap is visible at a glance.
    for tick, w in zip(ax.get_yticklabels(), words):
        is_shared = w in shared
        tick.set_color(INK if is_shared else INK_MUTED)
        tick.set_fontweight("bold" if is_shared else "normal")

fig.suptitle("Frequent words are nearly the same in both classes\n"
             + f"bold = also in the other class's top 12 ({len(shared)} of 12 overlap)",
             x=0.055, ha="left", fontsize=12, fontweight="bold", color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.86))
save(fig, "fig04_top_words.png")
plt.show()

**This is the argument for TF-IDF over raw counts, with evidence.** Eight of the top twelve
words are the same in both classes - `size`, `like`, `shoes`, `wear`, `just`, `fit`, `good`,
`bought`. They are frequent because the dataset is clothing and footwear reviews, not
because they indicate sentiment. A model fed raw frequency spends most of its capacity on
terms that cannot separate the classes.

TF-IDF's inverse-document-frequency term is exactly the fix: a term appearing in nearly
every review gets down-weighted toward zero, so what is left is the terms that actually
discriminate. Stop-word removal has already taken out `the`, `and`, `i`; this shows that it
does not go nearly far enough, because the uninformative-but-frequent words here are
domain-specific and no generic stop list contains them.

### 4.2 Which terms actually separate the classes?

The right question is not "which terms are frequent" but "which terms are much more likely
in one class than the other". We score each term by **log-odds ratio with add-one
smoothing**:

$$\text{score}(t) = \log \frac{c_{\text{pos}}(t) + 1}{N_{\text{pos}} + V} - \log \frac{c_{\text{neg}}(t) + 1}{N_{\text{neg}} + V}$$

Smoothing matters: without it, a term appearing 3 times in positives and 0 times in
negatives gets an infinite score, and the chart fills up with noise. Terms are restricted to
those seen at least 15 times so the ranking is not driven by accidents.

In [ ]:
V = len(vocab)
alpha = 1.0
log_odds = (
    np.log((pos_counts + alpha) / (pos_counts.sum() + alpha * V))
    - np.log((neg_counts + alpha) / (neg_counts.sum() + alpha * V))
)

MIN_TOTAL = 15
frequent_enough = (pos_counts + neg_counts) >= MIN_TOTAL
scores = np.where(frequent_enough, log_odds, 0.0)

k = 14
pos_idx = np.argsort(-scores)[:k][::-1]
neg_idx = np.argsort(scores)[:k][::-1]

print(f"terms scored: {frequent_enough.sum():,} of {V:,} (seen >= {MIN_TOTAL} times)\n")
print("most negative-leaning:", ", ".join(vocab[np.argsort(scores)[:14]]))
print("\nmost positive-leaning:", ", ".join(vocab[np.argsort(-scores)[:14]]))

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 5.4))

idx = np.concatenate([neg_idx, pos_idx])
vals = scores[idx]
ypos = np.arange(len(idx))
colors = [NEG if v < 0 else POS for v in vals]

ax.barh(ypos, vals, height=0.68, color=colors)
ax.axvline(0, color=AXIS, linewidth=1)

# Labels sit on the outside of each bar, so they never overlap the zero line.
for y_, v, word in zip(ypos, vals, vocab[idx]):
    offset = -0.09 if v < 0 else 0.09
    ax.text(v + offset, y_, word, va="center",
            ha="right" if v < 0 else "left", fontsize=9, color=INK)

ax.set_yticks([])
# Padding scaled to the longest label on each side ("extremely comfortable" is
# the one that overflows a symmetric margin).
pad_left = 0.22 * max(len(w) for w in vocab[neg_idx])
pad_right = 0.22 * max(len(w) for w in vocab[pos_idx])
ax.set_xlim(vals.min() - pad_left, vals.max() + pad_right)
ax.set_xlabel("log-odds ratio  (<- negative      positive ->)")
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)

handles = [plt.Line2D([], [], color=CLASS_COLOR[i], linewidth=8, label=CLASS_NAME[i])
           for i in (0, 1)]
ax.legend(handles=handles, title="leans toward", title_fontsize=9, loc="lower right")
ax.set_title("The terms that actually discriminate\n"
             + f"top {k} each way, terms seen >= {MIN_TOTAL} times, add-one smoothed", pad=10)

save(fig, "fig05_distinctive_words.png")
plt.show()

**Three things to use later.**

1. **Negative reviews are about the transaction, positives about the experience.** The
   negative side is dominated by `waste money`, `return item`, `returned`, `return policy`,
   `poor quality`, `worst`, `horrible`, `ripped`, `cheaply` - note how many concern
   *returning* the item rather than describing it. The positive side is `comfortable`,
   `highly recommend`, `loves`, `compliments`, `feel great`, `fits perfectly`. The strongest
   single signal that a review is negative is that the customer mentions sending it back.

2. **Bigrams are earning their place, and 4.3 shows it is not for the reason you would
   expect.** Of the top 50 negative terms, 7 are bigrams; of the top 50 positive terms, 22
   are. Phrases like `waste money`, `poor quality`, `highly recommend` and
   `extremely comfortable` carry meaning that neither word carries alone.

3. **Brand names leak in.** `nike` (+2.67, 56 occurrences) and `clarks` (+2.54, 49) both rank
   in the top 14 positive terms. That is a real correlation in this sample, not sentiment -
   this dataset's Nike and Clarks buyers happened to be happy ones. The model can learn
   "Nike means positive" instead of reading the review, which makes brand mentions a good
   thing to look for in the error analysis (e) and a genuine generalisation risk.

### 4.3 Does the bigram-negation argument actually hold?

Worth testing rather than asserting, since (a) has to justify `ngram_range=(1, 2)` and (e)
has to explain the model's weaknesses. The claim to check: bigrams rescue negation, because
`doesn work` becomes its own feature.

In [ ]:
NEGATION_CUES = ("not ", "no ", "never ", "doesn ", "didn ", "don ", "isn ",
                 "wasn ", "won ", "can ", "couldn ", "wouldn ")

neg_bigrams = [t for t in vocab if " " in t and t.startswith(NEGATION_CUES)]
nb_idx = np.array([int(np.where(vocab == t)[0][0]) for t in neg_bigrams])
nb_total = pos_counts[nb_idx] + neg_counts[nb_idx]

print(f"negation-cue bigrams kept in the 20k vocabulary: {len(neg_bigrams)}")
print(f"  of those, leaning negative: {int((log_odds[nb_idx] < 0).sum())}")
print(f"  median occurrences across the whole corpus: {np.median(nb_total):.0f}")
print(f"  passing the >= {MIN_TOTAL}-occurrence bar used in 4.2: "
      f"{int((nb_total >= MIN_TOTAL).sum())} of {len(neg_bigrams)}")

print("\nthe textbook examples, and how rare they actually are:")
rank_of = {t: r + 1 for r, t in enumerate(vocab[np.argsort(scores)])}
for term in ("doesn work", "didn fit", "doesn fit", "don like"):
    if term in rank_of:
        i = int(np.where(vocab == term)[0][0])
        total = int(pos_counts[i] + neg_counts[i])
        print(f"  {term:<12} {total:>3} occurrences  log-odds {log_odds[i]:+.2f}  "
              f"-> {'scored' if total >= MIN_TOTAL else 'EXCLUDED, too rare'}")

**The textbook argument half-works, and saying so is worth marks in (a) and (f).**

155 negation bigrams do survive into the vocabulary, so the mechanism exists. But they are
individually tiny: `doesn work` appears **7 times in 7,401 reviews**, `didn fit` 14 times,
and the median negation bigram occurs 5 times in the entire corpus. Only **11 of the 155**
are frequent enough to clear the modest 15-occurrence bar used in 4.2. And of all 155, just
80 even lean negative - 52%, barely better than a coin flip.

So bigrams help, but **not** mainly by fixing negation. They help by capturing fixed
multiword phrases - `waste money`, `highly recommend`, `poor quality` - which are frequent
enough to estimate a reliable weight for. Negation is left largely unsolved, because English
has far too many ways to negate for any of them to appear often enough to learn from. Every
unseen phrasing - "wouldn't buy again", "not as comfortable as I hoped" - falls straight
through.

**This is the argument for BERT, and we got it from the data rather than from a paper.**
A contextual model does not need to have seen "wouldn't buy again" as a unit; self-attention
lets the negation modify whatever it attaches to. Keep this paragraph for (f), and look for
a negation case in (e) that TF-IDF gets wrong and BERT gets right - the project plan is
right that this is the single most persuasive thing we can show.

## 5. Summary

Everything below is a measured number from this notebook, ready to be quoted in the report.

**The data**

| | |
|---|---|
| training reviews | 7,401 |
| test reviews (unlabelled) | 1,851 |
| class balance | 6,319 positive (85.4%) / 1,082 negative (14.6%) |
| median length | 42 words (negative 50, positive 41) |
| 99th percentile length | 269 words; longest 1,066 |
| repeated rows in train | 216 (151 distinct texts, no label conflicts) |
| reviews in both train and test | 87 |
| candidate 1-2 grams | 135,616 (13,820 unigrams only) |
| after `min_df=2` | 26,173 |
| final feature count (`max_features=20000`) | 20,000 - 14.7% of candidates |

**Decisions this justifies**

1. **Report macro F1 and per-class precision/recall, never accuracy alone.** Always-positive
   scores 85.4%. (figure `fig01`)
2. **`max_length = 256` for BERT.** Keeps 98.8% of reviews whole by word count, ~97.7% after
   subword expansion; 512 doubles sequence length for ~2% more coverage at roughly 4x the
   attention cost. (figure `fig02`)
3. **Keep bigrams - but for the measured reason, not the textbook one.** They pay off by
   capturing fixed phrases (22 of the top 50 positive terms are bigrams), not by fixing
   negation: `doesn work` occurs 7 times in the entire corpus, and of 155 negation bigrams
   only 11 are frequent enough to score and only 80 even lean negative. (section 4.3,
   figure `fig05`)
4. **TF-IDF rather than raw counts.** 8 of the 12 most frequent words are shared between the
   classes, so frequency alone is close to useless here; IDF is what suppresses them.
   (figure `fig04`)
5. **`class_weight="balanced"` in the baseline** follows directly from the 85/15 split.
6. **Length is not a label leak** - the two distributions overlap almost completely.
   (figure `fig03`)

**Caveats to state honestly in the report**

- Validation accuracy is mildly optimistic: duplicate reviews can straddle the 80/20 split.
- 87 test reviews are also in the training set, so ~4.7% of test predictions are recall
  rather than generalisation. This is a property of the given data; `data/` is read-only.
- `min_df=2` deliberately discards rare terms, which is a known weakness to probe in (e).

**Next:** Phase 2 - run `notebooks/colab_bert_training.ipynb` on a Colab T4 with
`max_length=256`.